# Ensemble Learning and AdaBoost

AdaBoost의 이진 분류 흐름을 명확하게 보기 위해 Iris 데이터 중 `versicolor`와 `virginica` 두 클래스만 사용하고, 시각화를 위해 꽃잎 길이와 꽃잎 너비 두 특성만 사용합니다.

### `#TODO` 부분에 코드를 채워 넣으세요


# Setup

실행 환경과 그래프 설정을 준비합니다.

In [ ]:
# 파이썬 버전이 예제 실행 조건을 만족하는지 확인합니다.
import sys

assert sys.version_info >= (3, 10)


In [ ]:
# scikit-learn 버전이 예제 실행 조건을 만족하는지 확인합니다.
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1")


In [ ]:
# 그래프의 글자 크기를 보기 좋게 설정합니다.
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)


In [ ]:
# 수치 계산과 격자 좌표 생성을 위해 NumPy를 불러옵니다.
import numpy as np


# AdaBoost Iris 데이터 준비

AdaBoost의 기본 원리를 쉽게 확인하기 위해 다음처럼 구성합니다.

- 입력 특성: 꽃잎 길이, 꽃잎 너비
- 분류 대상: versicolor, virginica 두 클래스
- 이유: 결정 경계를 2차원 그래프로 확인하기 쉽고, AdaBoost의 이진 분류 흐름을 명확하게 볼 수 있습니다.

In [ ]:
# load_iris: scikit-learn에 내장된 Iris 붓꽃 데이터셋을 불러오는 함수입니다.
from sklearn.datasets import load_iris

# train_test_split: 데이터를 학습용/테스트용으로 나누는 함수입니다.
from sklearn.model_selection import train_test_split

# SVC: 아래 수동 Boosting 시각화에서 약한 분류기처럼 반복 학습에 사용됩니다.
from sklearn.svm import SVC

# Iris 데이터셋을 불러옵니다.
iris = load_iris()

# 시각화를 위해 꽃잎 길이와 꽃잎 너비 두 특성만 사용합니다.
# iris.data[:, 2] = petal length, iris.data[:, 3] = petal width
X = iris.data[:, (2, 3)]
y = iris.target

# AdaBoost의 이진 분류 흐름을 보기 위해 setosa(0)는 제외하고,
# versicolor(1)와 virginica(2) 두 클래스만 사용합니다.
mask = y != 0
X = X[mask]
y = y[mask] - 1  # versicolor를 0, virginica를 1로 바꿉니다.

# 학습 데이터와 테스트 데이터를 나눕니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


# 결정 경계 시각화 함수 준비

Iris 데이터의 꽃잎 길이와 꽃잎 너비를 기준으로 AdaBoost의 결정 경계를 시각화합니다.


In [ ]:
# AdaBoost의 기본 약한 학습기로 사용할 결정 트리 분류기를 불러옵니다.
from sklearn.tree import DecisionTreeClassifier


In [ ]:
# 학습된 분류기의 결정 경계를 시각화하는 함수입니다.
def plot_decision_boundary(clf, X, y, alpha=1.0):
    # Iris 꽃잎 길이와 꽃잎 너비 범위에 맞게 그래프 축을 지정합니다.
    axes = [2.5, 7.5, 0.8, 2.8]

    # 2차원 평면 전체에 촘촘한 격자 좌표를 만듭니다.
    x1, x2 = np.meshgrid(np.linspace(axes[0], axes[1], 100),
                         np.linspace(axes[2], axes[3], 100))

    # 격자 좌표를 모델 입력 형태로 변환합니다.
    X_new = np.c_[x1.ravel(), x2.ravel()]

    # 각 격자점에 대해 모델이 예측한 클래스를 계산합니다.
    y_pred = clf.predict(X_new).reshape(x1.shape)

    # 예측 결과를 바탕으로 결정 영역을 색으로 표시합니다.
    plt.contourf(x1, x2, y_pred, alpha=0.3 * alpha, cmap='Wistia')
    plt.contour(x1, x2, y_pred, cmap="Greys", alpha=0.8 * alpha)

    # 실제 학습 데이터를 클래스별로 다른 마커로 표시합니다.
    colors = ["#78785c", "#c47b27"]
    markers = ("o", "^")
    labels = ["versicolor", "virginica"]
    for idx in (0, 1):
        plt.plot(X[:, 0][y == idx], X[:, 1][y == idx],
                 color=colors[idx], marker=markers[idx], linestyle="none",
                 label=labels[idx])

    # 축 범위와 축 이름을 설정합니다.
    plt.axis(axes)
    plt.xlabel("petal length")
    plt.ylabel("petal width", rotation=0)
    plt.legend(loc="upper left")


# Boosting
## AdaBoost

AdaBoost는 이전 학습기가 틀린 샘플에 더 큰 가중치를 주면서, 여러 약한 학습기를 순차적으로 학습시키는 앙상블 방법입니다.

In [ ]:
# extra code – this cell generates Figure 6–8
# AdaBoost의 핵심 아이디어를 수동으로 보여주는 시각화 코드입니다.
# 틀린 샘플의 가중치를 키우면서 분류기를 반복 학습합니다.

m = len(X_train)  # 학습 샘플 개수입니다.

fig, axes = plt.subplots(ncols=2, figsize=(10, 4), sharey=True)
for subplot, learning_rate in ((0, 1), (1, 0.5)):
    # 처음에는 모든 샘플에 동일한 가중치를 부여합니다.
    #TODO
    plt.sca(axes[subplot])

    # 분류기를 여러 번 반복 학습하면서 샘플 가중치를 갱신합니다.
    for i in range(5):
        svm_clf = SVC(C=0.2, gamma=0.6, random_state=42)
        svm_clf.fit(X_train, y_train, sample_weight=sample_weights * m)
        y_pred = svm_clf.predict(X_train)

        # 현재 분류기가 틀린 샘플들의 가중치 합을 계산합니다.
        error_weights = sample_weights[y_pred != y_train].sum()

        # 전체 가중치 중 오분류 가중치의 비율을 계산합니다.
        #TODO

        # 현재 분류기의 영향력을 계산합니다. 오차가 작을수록 alpha가 커집니다.
        alpha = learning_rate * np.log((1 - r) / r)  # equation 7-2

        # 틀린 샘플의 가중치를 증가시켜 다음 분류기가 더 집중하게 합니다.
        sample_weights[y_pred != y_train] *= np.exp(alpha)  # equation 7-3

        # 가중치의 합이 1이 되도록 정규화합니다.
        sample_weights /= sample_weights.sum()  # normalization step

        # 현재 단계의 결정 경계를 시각화합니다.
        plot_decision_boundary(svm_clf, X_train, y_train, alpha=0.4)
        plt.title(f"learning_rate = {learning_rate}")
    if subplot == 1:
        plt.ylabel("")

plt.show()


NameError: name 'X_train' is not defined

In [ ]:
# scikit-learn에서 제공하는 AdaBoost 분류기를 불러옵니다.
from sklearn.ensemble import AdaBoostClassifier

# AdaBoostClassifier를 생성합니다.
# DecisionTreeClassifier(max_depth=1)은 깊이가 1인 결정 트리이며, 흔히 decision stump라고 부릅니다.
# n_estimators=30은 약한 학습기 30개를 순차적으로 학습한다는 의미입니다.
# learning_rate=0.5는 각 학습기의 반영 정도를 조절합니다.
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=30,
    learning_rate=0.5, random_state=42)

# AdaBoost 모델을 Iris 학습 데이터에 맞춥니다.
ada_clf.fit(X_train, y_train)


In [ ]:
# extra code – in case you're curious to see what the decision boundary
#              looks like for the AdaBoost classifier
# Iris 데이터에서 학습된 AdaBoost 분류기의 결정 경계를 시각화합니다.
plot_decision_boundary(ada_clf, X_train, y_train)
plt.title("AdaBoost on Iris data")
plt.show()
